# Sleep Health Analysis & Risk Prediction
Predicting sleep disorder risk using **Apache PySpark** and **Machine Learning** in Google Colab.

## 1. Environment Setup
Install PySpark and initialize the Spark session.

In [ ]:
# Install PySpark in Google Colab
%pip install -q pyspark

In [ ]:
# Create and configure SparkSession
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SleepHealthMLProject")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("Spark session created successfully!")

## 2. Dataset Upload & Loading
Upload `sleep_health_dataset.csv` and load it into a PySpark DataFrame.

In [ ]:
# Upload the CSV dataset (skipped if file already exists in session storage)
import os
from google.colab import files

if not os.path.exists("sleep_health_dataset.csv"):
    print("Please select and upload your sleep_health_dataset.csv file:")
    uploaded = files.upload()

    for filename in uploaded:
        if filename.endswith(".csv"):
            os.rename(filename, "sleep_health_dataset.csv")

In [ ]:
# Load dataset into PySpark DataFrame
df = spark.read.csv(
    "sleep_health_dataset.csv",
    header=True,
    inferSchema=True
)

print("Dataset loaded successfully!")
print("Total Rows:", df.count())
print("Total Columns:", len(df.columns))

In [ ]:
# Preview first 5 rows
df.show(5, truncate=False)

In [ ]:
# View dataset schema
df.printSchema()

## 3. Exploratory Data Analysis (EDA)
Check missing values, class distributions, group summaries, and correlations.

In [ ]:
# Import EDA and visualization libraries
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

In [ ]:
# Check for missing values across all columns
total_rows = df.count()

missing = df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).first().asDict()

missing_cols = {c: n for c, n in missing.items() if n > 0}
print("Columns with missing values:", missing_cols if missing_cols else "None (Dataset is complete)")

### Target Variable Distribution

In [ ]:
# Target distribution: sleep_disorder_risk counts and percentages
risk_counts = (
    df.groupBy("sleep_disorder_risk")
    .count()
    .withColumn("Percentage",
                F.round(F.col("count") * 100 / total_rows, 2))
    .orderBy(F.desc("count"))
)

risk_counts.show()

In [ ]:
# Visualize risk category distribution
risk_data = risk_counts.collect()

plt.figure(figsize=(7, 4))
plt.bar(
    [r["sleep_disorder_risk"] for r in risk_data],
    [r["count"] for r in risk_data],
    color="steelblue",
    edgecolor="black"
)
plt.title("Sleep Disorder Risk Distribution")
plt.xlabel("Risk Category")
plt.ylabel("Number of Records")
plt.tight_layout()
plt.show()

### Summary Metrics by Risk Group

In [ ]:
# Average measurements for each risk group
summary = df.groupBy("sleep_disorder_risk").agg(
    F.avg("sleep_duration_hrs").alias("Sleep Hours"),
    F.avg("sleep_quality_score").alias("Sleep Quality"),
    F.avg("stress_score").alias("Stress"),
    F.avg("cognitive_performance_score").alias("Cognitive Score")
).orderBy("sleep_disorder_risk")

summary.show(truncate=False)

In [ ]:
# Visualize average measurements across risk groups
summary_data = summary.collect()
risk_names = [r["sleep_disorder_risk"] for r in summary_data]

graph_columns = [
    ("Sleep Hours", "Average Sleep Duration", "Hours"),
    ("Sleep Quality", "Average Sleep Quality", "Score"),
    ("Stress", "Average Stress by Risk Category", "Score"),
    ("Cognitive Score", "Average Cognitive Performance", "Score")
]

for column, title, ylabel in graph_columns:
    values = [float(r[column]) for r in summary_data]

    plt.figure(figsize=(7, 4))
    plt.bar(risk_names, values, color="teal", edgecolor="black")
    plt.title(title)
    plt.xlabel("Risk Category")
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()

### Feature Correlation Heatmap

In [ ]:
# Select numerical columns and compute Pearson correlation matrix
correlation_columns = [
    "sleep_duration_hrs",
    "sleep_quality_score",
    "stress_score",
    "cognitive_performance_score",
    "sleep_latency_mins",
    "wake_episodes_per_night",
    "rem_percentage",
    "deep_sleep_percentage"
]

correlation_data = df.select(*correlation_columns).dropna()

assembler = VectorAssembler(
    inputCols=correlation_columns,
    outputCol="correlation_features"
)

vector_data = assembler.transform(correlation_data)

matrix = Correlation.corr(
    vector_data, "correlation_features", "pearson"
).first()[0].toArray()

In [ ]:
# Plot correlation heatmap
plt.figure(figsize=(10, 8))
plt.imshow(matrix, cmap="coolwarm", vmin=-1, vmax=1)

plt.xticks(range(len(correlation_columns)),
           correlation_columns, rotation=90)
plt.yticks(range(len(correlation_columns)), correlation_columns)

for i in range(len(correlation_columns)):
    for j in range(len(correlation_columns)):
        plt.text(j, i, f"{float(matrix[i][j]):.2f}",
                 ha="center", va="center", fontsize=8)

plt.colorbar(label="Pearson Correlation")
plt.title("Selected Sleep Health Features - Correlation Heatmap")
plt.tight_layout()
plt.show()

## 4. Feature Selection & Train/Test Split
Select predictive features, separate categorical and numerical columns, and split into 80% train and 20% test sets.

In [ ]:
# Import ML modules
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [ ]:
# 1. Select input features
target = "sleep_disorder_risk"

excluded_columns = [
    "person_id",
    "sleep_disorder_risk",
    "cognitive_performance_score",
    "felt_rested"
]

feature_columns = [
    c for c in df.columns if c not in excluded_columns
]

categorical_columns = [
    c for c, dtype in df.dtypes
    if c in feature_columns and dtype == "string"
]

numerical_columns = [
    c for c, dtype in df.dtypes
    if c in feature_columns and dtype != "string"
]

print("Categorical columns:", categorical_columns)
print("Numerical columns:", numerical_columns)

# Remove records with missing input or target values
model_data = df.dropna(subset=feature_columns + [target])

In [ ]:
# 2. Split into training (80%) and testing (20%) data
train_data, test_data = model_data.randomSplit(
    [0.8, 0.2], seed=42
)

print("Training records:", train_data.count())
print("Testing records:", test_data.count())

## 5. Machine Learning Pipeline
Set up feature indexers, one-hot encoder, vector assembler, label indexer, and random forest classifier.

In [ ]:
# 3. StringIndexer and OneHotEncoder for categorical features
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_index",
        handleInvalid="keep"
    )
    for c in categorical_columns
]

encoder = OneHotEncoder(
    inputCols=[c + "_index" for c in categorical_columns],
    outputCols=[c + "_encoded" for c in categorical_columns],
    handleInvalid="keep"
)

# Combine numerical and encoded categorical features
encoded_columns = [c + "_encoded" for c in categorical_columns]

assembler = VectorAssembler(
    inputCols=numerical_columns + encoded_columns,
    outputCol="features",
    handleInvalid="skip"
)

# Convert target column into numerical labels
label_indexer = StringIndexer(
    inputCol=target,
    outputCol="label"
)

In [ ]:
# 4. Create Random Forest model and assemble the Pipeline
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50,
    maxDepth=8,
    seed=42
)

pipeline = Pipeline(
    stages=indexers + [
        encoder,
        assembler,
        label_indexer,
        rf
    ]
)

## 6. Model Training & Predictions
Fit the ML pipeline on the training data and make predictions on the test set.

In [ ]:
# Train the entire pipeline on training data
model = pipeline.fit(train_data)
print("Random Forest model trained successfully!")

In [ ]:
# Generate predictions on unseen test data
predictions = model.transform(test_data).cache()

# Preview sample predictions
predictions.select(
    target,
    "label",
    "prediction"
).show(10, truncate=False)

## 7. Model Evaluation & Performance Visualization
Calculate multiclass classification metrics (Accuracy, F1-Score, Precision, Recall) and plot them.

In [ ]:
# Evaluate model performance
metrics = [
    "accuracy",
    "f1",
    "weightedPrecision",
    "weightedRecall"
]

scores = {}

for metric in metrics:
    evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName=metric
    )

    scores[metric] = evaluator.evaluate(predictions)
    print(f"{metric:20s}: {scores[metric]:.4f}")

In [ ]:
# Plot model performance
plt.figure(figsize=(8, 4))
bars = plt.bar(
    list(scores.keys()),
    list(scores.values()),
    color="steelblue",
    edgecolor="black"
)
plt.ylim(0, 1.1)
plt.ylabel("Score")
plt.title("Random Forest Model Performance")
plt.xticks(rotation=15)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2.0, yval + 0.02, f"{yval:.4f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()